reconstructed subject IDs

In [2]:
# ============================================================
# ADD RECONSTRUCTED SUBJECT IDS TO PAIRED MANIFEST
# ============================================================
# Assumption:
# Within each distance group, the paired order index is consistent across
# all 7 gesture folders, so the same pair_index_within_group corresponds
# to the same subject for all gestures at that distance.
#
# Example:
# 4_feet + pair_index_within_group = 1  -> subject 4F_S001
# 4_feet + pair_index_within_group = 2  -> subject 4F_S002
# ...
# ============================================================

import json
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
cwd = Path.cwd()

if cwd.name == "notebooks":
    ROOT = cwd.parent
else:
    ROOT = cwd

MANIFEST_DIR = ROOT / "data" / "processed" / "manifests"
PAIRED_PATH = MANIFEST_DIR / "paired_manifest.csv"

print("ROOT:", ROOT)
print("PAIRED_PATH:", PAIRED_PATH)

if not PAIRED_PATH.exists():
    raise FileNotFoundError(f"paired_manifest.csv not found at: {PAIRED_PATH}")

# ------------------------------------------------------------
# Load paired manifest
# ------------------------------------------------------------
df = pd.read_csv(PAIRED_PATH)

print("Original manifest shape:", df.shape)
display(df.head())

required_cols = ["pair_id", "pair_index_within_group", "gesture", "distance"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# ------------------------------------------------------------
# Build reconstructed subject_id
# ------------------------------------------------------------
def make_subject_id(distance, pair_idx):
    if distance == "4_feet":
        prefix = "4F"
    elif distance == "6_feet":
        prefix = "6F"
    elif distance == "8_feet":
        prefix = "8F"
    else:
        prefix = "UNK"
    return f"{prefix}_S{int(pair_idx):03d}"

df["subject_id"] = df.apply(
    lambda row: make_subject_id(row["distance"], row["pair_index_within_group"]),
    axis=1
)

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------
print("\nUnique subject counts by distance:")
subject_counts = (
    df.groupby("distance")["subject_id"]
    .nunique()
    .reset_index(name="num_subjects")
)
display(subject_counts)

print("\nGesture count per reconstructed subject (should usually be 7 within a distance-group subject definition):")
subject_gesture_counts = (
    df.groupby(["distance", "subject_id"])["gesture"]
    .nunique()
    .reset_index(name="num_unique_gestures")
)
display(subject_gesture_counts.head(20))

bad_subjects = subject_gesture_counts[subject_gesture_counts["num_unique_gestures"] != 7]
print("\nSubjects not having exactly 7 gestures:", len(bad_subjects))
if len(bad_subjects) > 0:
    display(bad_subjects.head(20))

print("\nSample rows after subject_id reconstruction:")
display(df.head(20))

# ------------------------------------------------------------
# Save updated manifest
# ------------------------------------------------------------
UPDATED_PAIRED_PATH = MANIFEST_DIR / "paired_manifest_with_subjects.csv"
df.to_csv(UPDATED_PAIRED_PATH, index=False)

print("\nSaved updated paired manifest with subject IDs to:")
print(UPDATED_PAIRED_PATH)

# ------------------------------------------------------------
# Save short summary
# ------------------------------------------------------------
summary = {
    "total_rows": int(len(df)),
    "unique_subjects_total": int(df["subject_id"].nunique()),
    "subjects_by_distance": {
        row["distance"]: int(row["num_subjects"])
        for _, row in subject_counts.iterrows()
    },
    "subjects_with_all_7_gestures": int((subject_gesture_counts["num_unique_gestures"] == 7).sum()),
    "subjects_missing_any_gesture": int((subject_gesture_counts["num_unique_gestures"] != 7).sum()),
    "output_file": str(UPDATED_PAIRED_PATH),
}

summary_path = MANIFEST_DIR / "subject_reconstruction_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)

print("\nSummary:")
print(json.dumps(summary, indent=4))
print("Saved summary to:", summary_path)

ROOT: /data/Sajjan_Singh/gesture_recognition
PAIRED_PATH: /data/Sajjan_Singh/gesture_recognition/data/processed/manifests/paired_manifest.csv
Original manifest shape: (749, 26)


,pair_id,pair_index_within_group,subject_id,gesture,distance,capture_id,rgb_video_path,nir_video_path,rgb_num_frames,nir_num_frames,...,duration_diff,is_valid_pair,rgb_width,rgb_height,nir_width,nir_height,rgb_filename,nir_filename,rgb_serial_no,nir_serial_no
0,PAIR_00001,1,NaN,doctor,4_feet,1,/data/Sajjan_Singh/gesture_recognition/data/ra...,/data/Sajjan_Singh/gesture_recognition/data/ra...,184,135,...,0.812022,True,1080,1920,720,960,video_20260303_154701.mp4,20260303154701830.mp4,20260303154701,20260303154701830
1,PAIR_00002,2,NaN,doctor,4_feet,2,/data/Sajjan_Singh/gesture_recognition/data/ra...,/data/Sajjan_Singh/gesture_recognition/data/ra...,173,137,...,0.291978,True,1080,1920,720,960,video_20260303_155209.mp4,20260303155208958.mp4,20260303155209,20260303155208958
2,PAIR_00003,3,NaN,doctor,4_feet,3,/data/Sajjan_Singh/gesture_recognition/data/ra...,/data/Sajjan_Singh/gesture_recognition/data/ra...,180,137,...,0.518144,True,1080,1920,720,960,video_20260303_160149.mp4,20260303160149432.mp4,20260303160149,20260303160149432
3,PAIR_00004,4,NaN,doctor,4_feet,4,/data/Sajjan_Singh/gesture_recognition/data/ra...,/data/Sajjan_Singh/gesture_recognition/data/ra...,175,142,...,0.157589,True,1080,1920,720,960,video_20260303_161125.mp4,20260303161125449.mp4,20260303161125,20260303161125449
4,PAIR_00005,5,NaN,doctor,4_feet,5,/data/Sajjan_Singh/gesture_recognition/data/ra...,/data/Sajjan_Singh/gesture_recognition/data/ra...,167,135,...,0.173089,True,1080,1920,720,960,video_20260303_161400.mp4,20260303161400573.mp4,20260303161400,20260303161400573



Unique subject counts by distance:


,distance,num_subjects
0,4_feet,34
1,6_feet,35
2,8_feet,38



Gesture count per reconstructed subject (should usually be 7 within a distance-group subject definition):


,distance,subject_id,num_unique_gestures
0,4_feet,4F_S001,7
1,4_feet,4F_S002,7
2,4_feet,4F_S003,7
3,4_feet,4F_S004,7
4,4_feet,4F_S005,7
5,4_feet,4F_S006,7
6,4_feet,4F_S007,7
7,4_feet,4F_S008,7
8,4_feet,4F_S009,7
9,4_feet,4F_S010,7



Subjects not having exactly 7 gestures: 0

Sample rows after subject_id reconstruction:


,pair_id,pair_index_within_group,subject_id,gesture,distance,capture_id,rgb_video_path,nir_video_path,rgb_num_frames,nir_num_frames,...,duration_diff,is_valid_pair,rgb_width,rgb_height,nir_width,nir_height,rgb_filename,nir_filename,rgb_serial_no,nir_serial_no
0,PAIR_00001,1,4F_S001,doctor,4_feet,1,/data/Sajjan_Singh/gesture_recognition/data/ra...,/data/Sajjan_Singh/gesture_recognition/data/ra...,184,135,...,0.812022,True,1080,1920,720,960,video_20260303_154701.mp4,20260303154701830.mp4,20260303154701,20260303154701830
1,PAIR_00002,2,4F_S002,doctor,4_feet,2,/data/Sajjan_Singh/gesture_recognition/data/ra...,/data/Sajjan_Singh/gesture_recognition/data/ra...,173,137,...,0.291978,True,1080,1920,720,960,video_20260303_155209.mp4,20260303155208958.mp4,20260303155209,20260303155208958
2,PAIR_00003,3,4F_S003,doctor,4_feet,3,/data/Sajjan_Singh/gesture_recognition/data/ra...,/data/Sajjan_Singh/gesture_recognition/data/ra...,180,137,...,0.518144,True,1080,1920,720,960,video_20260303_160149.mp4,20260303160149432.mp4,20260303160149,20260303160149432
3,PAIR_00004,4,4F_S004,doctor,4_feet,4,/data/Sajjan_Singh/gesture_recognition/data/ra...,/data/Sajjan_Singh/gesture_recognition/data/ra...,175,142,...,0.157589,True,1080,1920,720,960,video_20260303_161125.mp4,20260303161125449.mp4,20260303161125,20260303161125449
4,PAIR_00005,5,4F_S005,doctor,4_feet,5,/data/Sajjan_Singh/gesture_recognition/data/ra...,/data/Sajjan_Singh/gesture_recognition/data/ra...,167,135,...,0.173089,True,1080,1920,720,960,video_20260303_161400.mp4,20260303161400573.mp4,20260303161400,20260303161400573
5,PAIR_00006,6,4F_S006,doctor,4_feet,6,/data/Sajjan_Singh/gesture_recognition/data/ra...,/data/Sajjan_Singh/gesture_recognition/data/ra...,172,129,...,0.541633,True,1080,1920,720,960,video_20260303_161659.mp4,20260303161659874.mp4,20260303161659,20260303161659874
6,PAIR_00007,7,4F_S007,doctor,4_feet,7,/data/Sajjan_Singh/gesture_recognition/data/ra...,/data/Sajjan_Singh/gesture_recognition/data/ra...,173,138,...,0.242978,True,1080,1920,720,960,video_20260303_162723.mp4,20260303162723511.mp4,20260303162723,20260303162723511
7,PAIR_00008,8,4F_S008,doctor,4_feet,8,/data/Sajjan_Singh/gesture_recognition/data/ra...,/data/Sajjan_Singh/gesture_recognition/data/ra...,176,123,...,0.948800,True,1080,1920,720,960,video_20260303_163030.mp4,20260303163030280.mp4,20260303163030,20260303163030280
8,PAIR_00009,9,4F_S009,doctor,4_feet,9,/data/Sajjan_Singh/gesture_recognition/data/ra...,/data/Sajjan_Singh/gesture_recognition/data/ra...,176,138,...,0.345800,True,1080,1920,720,960,video_20260303_163442.mp4,20260303163441824.mp4,20260303163442,20260303163441824
9,PAIR_00010,10,4F_S010,doctor,4_feet,10,/data/Sajjan_Singh/gesture_recognition/data/ra...,/data/Sajjan_Singh/gesture_recognition/data/ra...,168,129,...,0.395389,True,1080,1920,720,960,video_20260303_163947.mp4,20260303163947452.mp4,20260303163947,20260303163947452



Saved updated paired manifest with subject IDs to:
/data/Sajjan_Singh/gesture_recognition/data/processed/manifests/paired_manifest_with_subjects.csv

Summary:
{
    "total_rows": 749,
    "unique_subjects_total": 107,
    "subjects_by_distance": {
        "4_feet": 34,
        "6_feet": 35,
        "8_feet": 38
    },
    "subjects_with_all_7_gestures": 107,
    "subjects_missing_any_gesture": 0,
    "output_file": "/data/Sajjan_Singh/gesture_recognition/data/processed/manifests/paired_manifest_with_subjects.csv"
}
Saved summary to: /data/Sajjan_Singh/gesture_recognition/data/processed/manifests/subject_reconstruction_summary.json


In [3]:
# ============================================================
# 04_subject_split_creation.ipynb
# SUBJECT-AWARE SPLIT CREATION FOR RGB-NIR GESTURE DATASET
# ============================================================
# This notebook creates:
#
# Protocol A: Subject-independent split
#   - protocol_A_train.csv
#   - protocol_A_val.csv
#   - protocol_A_test.csv
#
# Protocol B: Cross-distance splits
#   - protocol_B_train_4_test_8.csv
#   - protocol_B_test_8.csv
#   - protocol_B_train_8_test_4.csv
#   - protocol_B_test_4.csv
#   - protocol_B_train_4_6_test_8.csv
#   - protocol_B_test_8_from_4_6.csv
#   - protocol_B_train_6_8_test_4.csv
#   - protocol_B_test_4_from_6_8.csv
#   - protocol_B_train_4_8_test_6.csv
#   - protocol_B_test_6_from_4_8.csv
#
# Also creates:
#   - protocol_debug_train.csv
#   - protocol_debug_val.csv
#   - protocol_debug_test.csv
#   - split_summary.json
#
# Input:
#   data/processed/manifests/paired_manifest_with_subjects.csv
# ============================================================

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

# ============================================================
# 1. PATHS
# ============================================================
cwd = Path.cwd()

if cwd.name == "notebooks":
    ROOT = cwd.parent
else:
    ROOT = cwd

MANIFEST_PATH = ROOT / "data" / "processed" / "manifests" / "paired_manifest_with_subjects.csv"
SPLIT_DIR = ROOT / "data" / "processed" / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print("Current working directory :", cwd)
print("Project ROOT              :", ROOT)
print("Manifest path             :", MANIFEST_PATH)
print("Split directory           :", SPLIT_DIR)

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"paired_manifest_with_subjects.csv not found at: {MANIFEST_PATH}")

# ============================================================
# 2. LOAD DATA
# ============================================================
df = pd.read_csv(MANIFEST_PATH)

print("\nLoaded manifest shape:", df.shape)
display(df.head())

required_cols = [
    "pair_id", "subject_id", "gesture", "distance",
    "rgb_video_path", "nir_video_path", "is_valid_pair"
]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Keep only valid pairs
df = df[df["is_valid_pair"] == True].copy().reset_index(drop=True)

print("\nValid rows after filtering:", len(df))
print("Unique subjects :", df["subject_id"].nunique())
print("Unique gestures :", sorted(df["gesture"].dropna().unique().tolist()))
print("Unique distances:", sorted(df["distance"].dropna().unique().tolist()))

# ============================================================
# 3. SANITY CHECKS
# ============================================================
print("\nCount by distance:")
display(df["distance"].value_counts().sort_index())

print("\nCount by gesture:")
display(df["gesture"].value_counts().sort_index())

print("\nCount by distance × gesture:")
display(pd.crosstab(df["distance"], df["gesture"]))

print("\nCount by subject (first 20):")
subject_counts = df["subject_id"].value_counts().sort_index()
display(subject_counts.head(20))

# Every reconstructed subject should have 7 gestures
subject_gesture_counts = (
    df.groupby("subject_id")["gesture"]
    .nunique()
    .reset_index(name="num_unique_gestures")
)
bad_subjects = subject_gesture_counts[subject_gesture_counts["num_unique_gestures"] != 7]

print("\nSubjects not having exactly 7 gestures:", len(bad_subjects))
if len(bad_subjects) > 0:
    display(bad_subjects)

# ============================================================
# 4. DEBUG RANDOM SPLIT (ROW-LEVEL, ONLY FOR PIPELINE DEBUGGING)
# ============================================================
# This split is not the main evaluation split.
# It is just useful for testing training code quickly.

SEED = 42
rng = np.random.default_rng(SEED)

df_debug = df.copy()
df_debug["strata"] = df_debug["gesture"].astype(str) + "__" + df_debug["distance"].astype(str)

train_parts = []
val_parts = []
test_parts = []

for strata_name, grp in df_debug.groupby("strata"):
    grp = grp.sample(frac=1, random_state=SEED).reset_index(drop=True)

    n = len(grp)
    n_train = int(round(0.70 * n))
    n_val = int(round(0.10 * n))
    n_test = n - n_train - n_val

    if n >= 3:
        n_train = max(1, n_train)
        n_val = max(1, n_val)
        n_test = n - n_train - n_val
        if n_test <= 0:
            n_test = 1
            if n_train > n_val:
                n_train -= 1
            else:
                n_val -= 1

    train_parts.append(grp.iloc[:n_train])
    val_parts.append(grp.iloc[n_train:n_train+n_val])
    test_parts.append(grp.iloc[n_train+n_val:])

debug_train = pd.concat(train_parts, ignore_index=True).drop(columns=["strata"])
debug_val = pd.concat(val_parts, ignore_index=True).drop(columns=["strata"])
debug_test = pd.concat(test_parts, ignore_index=True).drop(columns=["strata"])

debug_train.to_csv(SPLIT_DIR / "protocol_debug_train.csv", index=False)
debug_val.to_csv(SPLIT_DIR / "protocol_debug_val.csv", index=False)
debug_test.to_csv(SPLIT_DIR / "protocol_debug_test.csv", index=False)

print("\nSaved debug split:")
print(" - protocol_debug_train.csv :", len(debug_train))
print(" - protocol_debug_val.csv   :", len(debug_val))
print(" - protocol_debug_test.csv  :", len(debug_test))

# ============================================================
# 5. PROTOCOL A: SUBJECT-INDEPENDENT SPLIT
# ============================================================
# We split by subject_id, never by row.
# This ensures no subject leakage across train/val/test.

all_subjects = sorted(df["subject_id"].dropna().unique().tolist())
all_subjects = np.array(all_subjects)

rng = np.random.default_rng(SEED)
rng.shuffle(all_subjects)

n_subjects = len(all_subjects)
n_train_subj = int(round(0.70 * n_subjects))
n_val_subj = int(round(0.10 * n_subjects))
n_test_subj = n_subjects - n_train_subj - n_val_subj

if n_subjects >= 3:
    n_train_subj = max(1, n_train_subj)
    n_val_subj = max(1, n_val_subj)
    n_test_subj = n_subjects - n_train_subj - n_val_subj
    if n_test_subj <= 0:
        n_test_subj = 1
        if n_train_subj > n_val_subj:
            n_train_subj -= 1
        else:
            n_val_subj -= 1

train_subjects = set(all_subjects[:n_train_subj])
val_subjects = set(all_subjects[n_train_subj:n_train_subj+n_val_subj])
test_subjects = set(all_subjects[n_train_subj+n_val_subj:])

protocol_A_train = df[df["subject_id"].isin(train_subjects)].copy().reset_index(drop=True)
protocol_A_val = df[df["subject_id"].isin(val_subjects)].copy().reset_index(drop=True)
protocol_A_test = df[df["subject_id"].isin(test_subjects)].copy().reset_index(drop=True)

protocol_A_train.to_csv(SPLIT_DIR / "protocol_A_train.csv", index=False)
protocol_A_val.to_csv(SPLIT_DIR / "protocol_A_val.csv", index=False)
protocol_A_test.to_csv(SPLIT_DIR / "protocol_A_test.csv", index=False)

print("\nSaved Protocol A subject-independent split:")
print(" - protocol_A_train.csv :", len(protocol_A_train), "| subjects:", len(train_subjects))
print(" - protocol_A_val.csv   :", len(protocol_A_val), "| subjects:", len(val_subjects))
print(" - protocol_A_test.csv  :", len(protocol_A_test), "| subjects:", len(test_subjects))

# Verify no subject leakage
train_subj_overlap_val = len(train_subjects.intersection(val_subjects))
train_subj_overlap_test = len(train_subjects.intersection(test_subjects))
val_subj_overlap_test = len(val_subjects.intersection(test_subjects))

print("\nProtocol A subject overlap checks:")
print(" Train ∩ Val  :", train_subj_overlap_val)
print(" Train ∩ Test :", train_subj_overlap_test)
print(" Val ∩ Test   :", val_subj_overlap_test)

# ============================================================
# 6. PROTOCOL B: CROSS-DISTANCE SPLITS
# ============================================================
protocol_b_summary = []

def save_split(train_df, test_df, train_name, test_name):
    train_path = SPLIT_DIR / train_name
    test_path = SPLIT_DIR / test_name
    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)
    return str(train_path), str(test_path)

def make_distance_split(train_distances, test_distances, train_file, test_file, split_name):
    train_df = df[df["distance"].isin(train_distances)].copy().reset_index(drop=True)
    test_df = df[df["distance"].isin(test_distances)].copy().reset_index(drop=True)

    train_path, test_path = save_split(train_df, test_df, train_file, test_file)

    protocol_b_summary.append({
        "split_name": split_name,
        "train_distances": train_distances,
        "test_distances": test_distances,
        "train_count": int(len(train_df)),
        "test_count": int(len(test_df)),
        "train_subjects": int(train_df["subject_id"].nunique()),
        "test_subjects": int(test_df["subject_id"].nunique()),
        "train_path": train_path,
        "test_path": test_path,
    })

    print(f"\nSaved {split_name}")
    print(" Train distances:", train_distances, "| count:", len(train_df), "| subjects:", train_df['subject_id'].nunique())
    print(" Test distances :", test_distances, "| count:", len(test_df), "| subjects:", test_df['subject_id'].nunique())

# train 4 -> test 8
make_distance_split(
    train_distances=["4_feet"],
    test_distances=["8_feet"],
    train_file="protocol_B_train_4_test_8.csv",
    test_file="protocol_B_test_8.csv",
    split_name="Protocol B: train 4 -> test 8"
)

# train 8 -> test 4
make_distance_split(
    train_distances=["8_feet"],
    test_distances=["4_feet"],
    train_file="protocol_B_train_8_test_4.csv",
    test_file="protocol_B_test_4.csv",
    split_name="Protocol B: train 8 -> test 4"
)

# train 4+6 -> test 8
make_distance_split(
    train_distances=["4_feet", "6_feet"],
    test_distances=["8_feet"],
    train_file="protocol_B_train_4_6_test_8.csv",
    test_file="protocol_B_test_8_from_4_6.csv",
    split_name="Protocol B: train 4+6 -> test 8"
)

# train 6+8 -> test 4
make_distance_split(
    train_distances=["6_feet", "8_feet"],
    test_distances=["4_feet"],
    train_file="protocol_B_train_6_8_test_4.csv",
    test_file="protocol_B_test_4_from_6_8.csv",
    split_name="Protocol B: train 6+8 -> test 4"
)

# train 4+8 -> test 6
make_distance_split(
    train_distances=["4_feet", "8_feet"],
    test_distances=["6_feet"],
    train_file="protocol_B_train_4_8_test_6.csv",
    test_file="protocol_B_test_6_from_4_8.csv",
    split_name="Protocol B: train 4+8 -> test 6"
)

# same-distance reference splits
make_distance_split(
    train_distances=["4_feet"],
    test_distances=["4_feet"],
    train_file="protocol_B_train_4_test_4.csv",
    test_file="protocol_B_test_4_same.csv",
    split_name="Protocol B reference: train 4 -> test 4"
)

make_distance_split(
    train_distances=["6_feet"],
    test_distances=["6_feet"],
    train_file="protocol_B_train_6_test_6.csv",
    test_file="protocol_B_test_6_same.csv",
    split_name="Protocol B reference: train 6 -> test 6"
)

make_distance_split(
    train_distances=["8_feet"],
    test_distances=["8_feet"],
    train_file="protocol_B_train_8_test_8.csv",
    test_file="protocol_B_test_8_same.csv",
    split_name="Protocol B reference: train 8 -> test 8"
)

# ============================================================
# 7. PROTOCOL C NOTE
# ============================================================
protocol_c_note = (
    "A distinct Protocol C (subject-independent + cross-distance) is not separately created here, "
    "because reconstructed subject IDs are already distance-specific. "
    "In this dataset, Protocol B cross-distance evaluation is effectively cross-subject as well."
)

print("\nProtocol C note:")
print(protocol_c_note)

# ============================================================
# 8. SAVE SUMMARY JSON
# ============================================================
summary = {
    "manifest_total_valid_pairs": int(len(df)),
    "unique_subjects": int(df["subject_id"].nunique()),
    "unique_gestures": sorted(df["gesture"].dropna().unique().tolist()),
    "unique_distances": sorted(df["distance"].dropna().unique().tolist()),
    "protocol_debug": {
        "train_count": int(len(debug_train)),
        "val_count": int(len(debug_val)),
        "test_count": int(len(debug_test)),
        "train_file": str(SPLIT_DIR / "protocol_debug_train.csv"),
        "val_file": str(SPLIT_DIR / "protocol_debug_val.csv"),
        "test_file": str(SPLIT_DIR / "protocol_debug_test.csv"),
        "note": "For debugging/training pipeline only. Not the main benchmark."
    },
    "protocol_A": {
        "train_count": int(len(protocol_A_train)),
        "val_count": int(len(protocol_A_val)),
        "test_count": int(len(protocol_A_test)),
        "train_subjects": int(len(train_subjects)),
        "val_subjects": int(len(val_subjects)),
        "test_subjects": int(len(test_subjects)),
        "subject_overlap_train_val": int(train_subj_overlap_val),
        "subject_overlap_train_test": int(train_subj_overlap_test),
        "subject_overlap_val_test": int(val_subj_overlap_test),
        "train_file": str(SPLIT_DIR / "protocol_A_train.csv"),
        "val_file": str(SPLIT_DIR / "protocol_A_val.csv"),
        "test_file": str(SPLIT_DIR / "protocol_A_test.csv"),
    },
    "protocol_B_splits": protocol_b_summary,
    "protocol_C_note": protocol_c_note,
}

summary_path = SPLIT_DIR / "split_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)

print("\nSaved split summary to:")
print(summary_path)

print("\n========== FINAL SPLIT SUMMARY ==========")
print(json.dumps(summary, indent=4))

# ============================================================
# 9. QUICK PREVIEW TABLES
# ============================================================
print("\nProtocol A: counts by distance")
display({
    "train": protocol_A_train["distance"].value_counts().sort_index().to_dict(),
    "val": protocol_A_val["distance"].value_counts().sort_index().to_dict(),
    "test": protocol_A_test["distance"].value_counts().sort_index().to_dict(),
})

print("\nProtocol A: counts by gesture")
display({
    "train": protocol_A_train["gesture"].value_counts().sort_index().to_dict(),
    "val": protocol_A_val["gesture"].value_counts().sort_index().to_dict(),
    "test": protocol_A_test["gesture"].value_counts().sort_index().to_dict(),
})

print("\nProtocol B summary table")
display(pd.DataFrame(protocol_b_summary))

print("\nSaved files in split directory:")
for p in sorted(SPLIT_DIR.glob("*.csv")):
    print(" -", p.name)

Current working directory : /data/Sajjan_Singh/gesture_recognition/notebooks
Project ROOT              : /data/Sajjan_Singh/gesture_recognition
Manifest path             : /data/Sajjan_Singh/gesture_recognition/data/processed/manifests/paired_manifest_with_subjects.csv
Split directory           : /data/Sajjan_Singh/gesture_recognition/data/processed/splits

Loaded manifest shape: (749, 26)


,pair_id,pair_index_within_group,subject_id,gesture,distance,capture_id,rgb_video_path,nir_video_path,rgb_num_frames,nir_num_frames,rgb_fps,nir_fps,duration_rgb,duration_nir,fps_diff,frame_diff,duration_diff,is_valid_pair,rgb_width,rgb_height,nir_width,nir_height,rgb_filename,nir_filename,rgb_serial_no,nir_serial_no
0,PAIR_00001,1,4F_S001,doctor,4_feet,1,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_154701.mp4,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303154701830.mp4,184,135,29.696472,25.074294,6.196022,5.384,4.622178,49,0.812022,True,1080,1920,720,960,video_20260303_154701.mp4,20260303154701830.mp4,20260303154701,20260303154701830
1,PAIR_00002,2,4F_S002,doctor,4_feet,2,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_155209.mp4,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303155208958.mp4,173,137,30.019203,25.041126,5.762978,5.471,4.978077,36,0.291978,True,1080,1920,720,960,video_20260303_155209.mp4,20260303155208958.mp4,20260303155209,20260303155208958
2,PAIR_00003,3,4F_S003,doctor,4_feet,3,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_160149.mp4,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303160149432.mp4,180,137,30.019290,25.009127,5.996144,5.478,5.010163,43,0.518144,True,1080,1920,720,960,video_20260303_160149.mp4,20260303160149432.mp4,20260303160149,20260303160149432
3,PAIR_00004,4,4F_S004,doctor,4_feet,4,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_161125.mp4,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303161125449.mp4,175,142,30.019270,25.035261,5.829589,5.672,4.984009,33,0.157589,True,1080,1920,720,960,video_20260303_161125.mp4,20260303161125449.mp4,20260303161125,20260303161125449
4,PAIR_00005,5,4F_S005,doctor,4_feet,5,/data/Sajjan_Singh/gesture_recognition/data/raw/RGB/4_feet/doctor/video_20260303_161400.mp4,/data/Sajjan_Singh/gesture_recognition/data/raw/NIR/nir/4_feet/doctor/20260303161400573.mp4,167,135,30.019294,25.046382,5.563089,5.390,4.972912,32,0.173089,True,1080,1920,720,960,video_20260303_161400.mp4,20260303161400573.mp4,20260303161400,20260303161400573



Valid rows after filtering: 749
Unique subjects : 107
Unique gestures : ['doctor', 'emergency', 'fire', 'help', 'robbery', 'sit_down', 'stand_up']
Unique distances: ['4_feet', '6_feet', '8_feet']

Count by distance:


distance
4_feet    238
6_feet    245
8_feet    266
Name: count, dtype: int64


Count by gesture:


...


Protocol A: counts by gesture


{'train': {'doctor': 75,
  'emergency': 75,
  'fire': 75,
  'help': 75,
  'robbery': 75,
  'sit_down': 75,
  'stand_up': 75},
 'val': {'doctor': 11,
  'emergency': 11,
  'fire': 11,
  'help': 11,
  'robbery': 11,
  'sit_down': 11,
  'stand_up': 11},
 'test': {'doctor': 21,
  'emergency': 21,
  'fire': 21,
  'help': 21,
  'robbery': 21,
  'sit_down': 21,
  'stand_up': 21}}


Protocol B summary table


,split_name,train_distances,test_distances,train_count,test_count,train_subjects,test_subjects,train_path,test_path
0,Protocol B: train 4 -> test 8,[4_feet],[8_feet],238,266,34,38,/data/Sajjan_Singh/gesture_recognition/data/processed/splits/protocol_B_train_4_test_8.csv,/data/Sajjan_Singh/gesture_recognition/data/processed/splits/protocol_B_test_8.csv
1,Protocol B: train 8 -> test 4,[8_feet],[4_feet],266,238,38,34,/data/Sajjan_Singh/gesture_recognition/data/processed/splits/protocol_B_train_8_test_4.csv,/data/Sajjan_Singh/gesture_recognition/data/processed/splits/protocol_B_test_4.csv
2,Protocol B: train 4+6 -> test 8,"[4_feet, 6_feet]",[8_feet],483,266,69,38,/data/Sajjan_Singh/gesture_recognition/data/processed/splits/protocol_B_train_4_6_test_8.csv,/data/Sajjan_Singh/gesture_recognition/data/processed/splits/protocol_B_test_8_from_4_6.csv
3,Protocol B: train 6+8 -> test 4,"[6_feet, 8_feet]",[4_feet],511,238,73,34,/data/Sajjan_Singh/gesture_recognition/data/processed/splits/protocol_B_train_6_8_test_4.csv,/data/Sajjan_Singh/gesture_recognition/data/processed/splits/protocol_B_test_4_from_6_8.csv
4,Protocol B: train 4+8 -> test 6,"[4_feet, 8_feet]",[6_feet],504,245,72,35,/data/Sajjan_Singh/gesture_recognition/data/processed/splits/protocol_B_train_4_8_test_6.csv,/data/Sajjan_Singh/gesture_recognition/data/processed/splits/protocol_B_test_6_from_4_8.csv
5,Protocol B reference: train 4 -> test 4,[4_feet],[4_feet],238,238,34,34,/data/Sajjan_Singh/gesture_recognition/data/processed/splits/protocol_B_train_4_test_4.csv,/data/Sajjan_Singh/gesture_recognition/data/processed/splits/protocol_B_test_4_same.csv
6,Protocol B reference: train 6 -> test 6,[6_feet],[6_feet],245,245,35,35,/data/Sajjan_Singh/gesture_recognition/data/processed/splits/protocol_B_train_6_test_6.csv,/data/Sajjan_Singh/gesture_recognition/data/processed/splits/protocol_B_test_6_same.csv
7,Protocol B reference: train 8 -> test 8,[8_feet],[8_feet],266,266,38,38,/data/Sajjan_Singh/gesture_recognition/data/processed/splits/protocol_B_train_8_test_8.csv,/data/Sajjan_Singh/gesture_recognition/data/processed/splits/protocol_B_test_8_same.csv



Saved files in split directory:
 - protocol_A_test.csv
 - protocol_A_train.csv
 - protocol_A_val.csv
 - protocol_B_test_4.csv
 - protocol_B_test_4_from_6_8.csv
 - protocol_B_test_4_same.csv
 - protocol_B_test_6_from_4_8.csv
 - protocol_B_test_6_same.csv
 - protocol_B_test_8.csv
 - protocol_B_test_8_from_4_6.csv
 - protocol_B_test_8_same.csv
 - protocol_B_train_4_6_test_8.csv
 - protocol_B_train_4_8_test_6.csv
 - protocol_B_train_4_test_4.csv
 - protocol_B_train_4_test_8.csv
 - protocol_B_train_6_8_test_4.csv
 - protocol_B_train_6_test_6.csv
 - protocol_B_train_8_test_4.csv
 - protocol_B_train_8_test_8.csv
 - protocol_debug_test.csv
 - protocol_debug_train.csv
 - protocol_debug_val.csv
